## एम्बेडिंग्स

हमारे पिछले उदाहरण में, हमने उच्च-आयामी बैग-ऑफ-शब्द वेक्टरों के साथ संचालन किया, जिनकी लंबाई `vocab_size` थी, और हम स्पष्ट रूप से नीच-आयामी स्थानिक प्रतिनिधित्व वेक्टरों को विरल वन-हॉट प्रतिनिधित्व में परिवर्तित कर रहे थे। यह वन-हॉट प्रतिनिधित्व मेमोरी के लिहाज से कुशल नहीं है, इसके अलावा, प्रत्येक शब्द को एक दूसरे से स्वतंत्र रूप से माना जाता है, अर्थात वन-हॉट एन्कोड किए गए वेक्टर शब्दों के बीच किसी भी सार्थक समानता को प्रकट नहीं करते हैं।

इस इकाई में, हम **News AG** डेटासेट का अन्वेषण जारी रखेंगे। शुरू करने के लिए, आइए डेटा लोड करें और पिछले नोटबुक से कुछ परिभाषाएँ लें।


In [1]:
import torch
import torchtext
import numpy as np
from torchnlp import *
train_dataset, test_dataset, classes, vocab = load_dataset()
vocab_size = len(vocab)
print("Vocab size = ",vocab_size)

Loading dataset...


d:\WORK\ai-for-beginners\5-NLP\14-Embeddings\data\train.csv: 29.5MB [00:01, 18.8MB/s]                            
d:\WORK\ai-for-beginners\5-NLP\14-Embeddings\data\test.csv: 1.86MB [00:00, 11.2MB/s]                          


Building vocab...
Vocab size =  95812



## एम्बेडिंग क्या है?

**एम्बेडिंग** का विचार यह है कि शब्दों को कम-आयामी घने वेक्टर द्वारा प्रस्तुत किया जाए, जो किसी तरह से शब्द के अर्थ को दर्शाते हैं। हम बाद में चर्चा करेंगे कि अर्थपूर्ण शब्द एम्बेडिंग कैसे बनाएं, लेकिन अभी के लिए हम एम्बेडिंग को शब्द वेक्टर के आयाम को कम करने के एक तरीके के रूप में सोचते हैं।

तो, एम्बेडिंग लेयर एक शब्द को इनपुट के रूप में लेगा, और निर्दिष्ट `embedding_size` का आउटपुट वेक्टर बनाएगा। किसी नज़रिए से, यह `Linear` लेयर के बहुत समान है, लेकिन एक-हॉट एन्कोडेड वेक्टर लेने के बजाय, यह शब्द संख्या को इनपुट के रूप में लेने में सक्षम होगा।

हमारे नेटवर्क में पहले लेयर के रूप में एम्बेडिंग लेयर का उपयोग करके, हम बैग-ऑफ-शब्द से **एम्बेडिंग बैग** मॉडल में स्विच कर सकते हैं, जहां हम पहले हमारे टेक्स्ट के प्रत्येक शब्द को संबंधित एम्बेडिंग में बदलते हैं, और फिर उन सभी एम्बेडिंग्स पर कुछ समेकित फ़ंक्शन जैसे कि `sum`, `average` या `max` की गणना करते हैं।

![पांच अनुक्रम शब्दों के लिए एक एम्बेडिंग क्लासिफायर दिखाती छवि.](../../../../../translated_images/hi/embedding-classifier-example.b77f021a7ee67eee.webp)

हमारा क्लासिफायर न्यूरल नेटवर्क एम्बेडिंग लेयर के साथ शुरू होगा, फिर एग्रीगेशन लेयर, और उसके ऊपर एक रैखिक क्लासिफायर होगा:


In [2]:
class EmbedClassifier(torch.nn.Module):
    def __init__(self, vocab_size, embed_dim, num_class):
        super().__init__()
        self.embedding = torch.nn.Embedding(vocab_size, embed_dim)
        self.fc = torch.nn.Linear(embed_dim, num_class)

    def forward(self, x):
        x = self.embedding(x)
        x = torch.mean(x,dim=1)
        return self.fc(x)

### चर अनुक्रम आकार के साथ निपटना

इस वास्तुकला के परिणामस्वरूप, हमारे नेटवर्क के लिए मिनिबैच विशेष तरीके से बनाए जाने होंगे। पिछले यूनिट में, जब बो-ऑफ़-वर्ड का उपयोग किया गया था, तो मिनिबैच में सभी BoW टेन्सर का आकार समान होता था `vocab_size`, हमारे पाठ अनुक्रम की वास्तविक लंबाई की परवाह किए बिना। जब हम शब्द एम्बेडिंग्स की ओर बढ़ेंगे, तो प्रत्येक पाठ नमूने में शब्दों की संख्या बदलती रहेगी, और जब उन नमूनों को मिनिबैच में मिलाएंगे तो हमें कुछ पैडिंग लागू करनी होगी।

यह उसी तकनीक का उपयोग करके किया जा सकता है, जहाँ डेटा स्रोत को `collate_fn` फ़ंक्शन प्रदान किया जाता है:


In [3]:
def padify(b):
    # b is the list of tuples of length batch_size
    #   - first element of a tuple = label, 
    #   - second = feature (text sequence)
    # build vectorized sequence
    v = [encode(x[1]) for x in b]
    # first, compute max length of a sequence in this minibatch
    l = max(map(len,v))
    return ( # tuple of two tensors - labels and features
        torch.LongTensor([t[0]-1 for t in b]),
        torch.stack([torch.nn.functional.pad(torch.tensor(t),(0,l-len(t)),mode='constant',value=0) for t in v])
    )

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=16, collate_fn=padify, shuffle=True)

### एम्बेडिंग क्लासिफायर प्रशिक्षण

अब जब हमने उचित डाटालोडर परिभाषित कर लिया है, तो हम उस प्रशिक्षण फ़ंक्शन का उपयोग करके मॉडल को प्रशिक्षित कर सकते हैं जिसे हमने पिछले यूनिट में परिभाषित किया था:


In [4]:
net = EmbedClassifier(vocab_size,32,len(classes)).to(device)
train_epoch(net,train_loader, lr=1, epoch_size=25000)

3200: acc=0.6415625
6400: acc=0.6865625
9600: acc=0.7103125
12800: acc=0.726953125
16000: acc=0.739375
19200: acc=0.75046875
22400: acc=0.7572321428571429


(0.889799795315499, 0.7623160588611644)

> **नोट**: हम यहां केवल 25k रिकॉर्ड्स के लिए प्रशिक्षण दे रहे हैं (एक पूर्ण युग से कम) समय की बचत के लिए, लेकिन आप प्रशिक्षण जारी रख सकते हैं, कई युगों के लिए प्रशिक्षण देने के लिए एक फ़ंक्शन लिख सकते हैं, और उच्च सटीकता प्राप्त करने के लिए लर्निंग रेट पैरामीटर के साथ प्रयोग कर सकते हैं। आपको लगभग 90% की सटीकता तक पहुंचने में सक्षम होना चाहिए।


### एम्बेडिंगबैग लेयर और परिवर्तनीय लंबाई अनुक्रम प्रतिनिधित्व

पिछले आर्किटेक्चर में, हमें सभी अनुक्रमों को एक ही लंबाई में पैड करना पड़ता था ताकि उन्हें एक मिनीबैच में फिट किया जा सके। यह परिवर्तनीय लंबाई अनुक्रमों का प्रतिनिधित्व करने का सबसे कुशल तरीका नहीं है - एक अन्य तरीका होगा **offset** वेक्टर का उपयोग करना, जो एक बड़े वेक्टर में संग्रहित सभी अनुक्रमों के ऑफसेट रखेगा।

![एक ऑफसेट अनुक्रम प्रतिनिधित्व दिखाने वाली छवि](../../../../../translated_images/hi/offset-sequence-representation.eb73fcefb29b46ee.webp)

> **नोट**: ऊपर की तस्वीर में, हमने वर्णों का एक अनुक्रम दिखाया है, लेकिन हमारे उदाहरण में हम शब्दों के अनुक्रम के साथ काम कर रहे हैं। हालांकि, ऑफसेट वेक्टर के साथ अनुक्रमों का प्रतिनिधित्व करने का सामान्य सिद्धांत वही रहता है।

ऑफसेट प्रतिनिधित्व के साथ काम करने के लिए, हम [`EmbeddingBag`](https://pytorch.org/docs/stable/generated/torch.nn.EmbeddingBag.html) लेयर का उपयोग करते हैं। यह `Embedding` के समान है, लेकिन यह कंटेंट वेक्टर और ऑफसेट वेक्टर को इनपुट के रूप में लेता है, और इसमें औसत निकालने वाली लेयर भी शामिल होती है, जो कि `mean`, `sum` या `max` हो सकती है।

यहाँ एक संशोधित नेटवर्क है जो `EmbeddingBag` का उपयोग करता है:


In [5]:
class EmbedClassifier(torch.nn.Module):
    def __init__(self, vocab_size, embed_dim, num_class):
        super().__init__()
        self.embedding = torch.nn.EmbeddingBag(vocab_size, embed_dim)
        self.fc = torch.nn.Linear(embed_dim, num_class)

    def forward(self, text, off):
        x = self.embedding(text, off)
        return self.fc(x)

प्रशिक्षण के लिए डेटासेट तैयार करने के लिए, हमें एक रूपांतरण फ़ंक्शन प्रदान करने की आवश्यकता है जो ऑफसेट वेक्टर तैयार करेगा:


In [6]:
def offsetify(b):
    # first, compute data tensor from all sequences
    x = [torch.tensor(encode(t[1])) for t in b]
    # now, compute the offsets by accumulating the tensor of sequence lengths
    o = [0] + [len(t) for t in x]
    o = torch.tensor(o[:-1]).cumsum(dim=0)
    return ( 
        torch.LongTensor([t[0]-1 for t in b]), # labels
        torch.cat(x), # text 
        o
    )

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=16, collate_fn=offsetify, shuffle=True)

ध्यान दें, कि सभी पिछले उदाहरणों के विपरीत, हमारा नेटवर्क अब दो पैरामीटर स्वीकार करता है: डेटा वेक्टर और ऑफसेट वेक्टर, जो विभिन्न आकार के होते हैं। इसी तरह, हमारा डेटा लोडर भी 2 के बजाय 3 मान प्रदान करता है: दोनों टेक्स्ट और ऑफसेट वेक्टर फीचर्स के रूप में प्रदान किए जाते हैं। इसलिए, हमें अपनी प्रशिक्षण फ़ंक्शन को थोड़ा समायोजित करने की आवश्यकता है ताकि इसका ध्यान रखा जा सके:


In [7]:
net = EmbedClassifier(vocab_size,32,len(classes)).to(device)

def train_epoch_emb(net,dataloader,lr=0.01,optimizer=None,loss_fn = torch.nn.CrossEntropyLoss(),epoch_size=None, report_freq=200):
    optimizer = optimizer or torch.optim.Adam(net.parameters(),lr=lr)
    loss_fn = loss_fn.to(device)
    net.train()
    total_loss,acc,count,i = 0,0,0,0
    for labels,text,off in dataloader:
        optimizer.zero_grad()
        labels,text,off = labels.to(device), text.to(device), off.to(device)
        out = net(text, off)
        loss = loss_fn(out,labels) #cross_entropy(out,labels)
        loss.backward()
        optimizer.step()
        total_loss+=loss
        _,predicted = torch.max(out,1)
        acc+=(predicted==labels).sum()
        count+=len(labels)
        i+=1
        if i%report_freq==0:
            print(f"{count}: acc={acc.item()/count}")
        if epoch_size and count>epoch_size:
            break
    return total_loss.item()/count, acc.item()/count


train_epoch_emb(net,train_loader, lr=4, epoch_size=25000)

3200: acc=0.6153125
6400: acc=0.6615625
9600: acc=0.6932291666666667
12800: acc=0.715078125
16000: acc=0.7270625
19200: acc=0.7382291666666667
22400: acc=0.7486160714285715


(22.771553103007037, 0.7551983365323096)


## सेमांटिक एम्बेडिंग्स: Word2Vec

हमारे पिछले उदाहरण में, मॉडल एम्बेडिंग लेयर ने शब्दों को वेक्टर प्रतिनिधित्व में मैप करना सीखा था, हालांकि, इस प्रतिनिधित्व का ज्यादा सैमान्टिक मतलब नहीं था। ऐसा वेक्टर प्रतिनिधित्व सीखना अच्छा होगा, जिसमें समान शब्द या पर्यायवाची वेक्टर ऐसे हों जो कुछ वेक्टर दूरी (जैसे युक्लिडियन दूरी) के दृष्टिकोण से एक-दूसरे के करीब हों।

ऐसा करने के लिए, हमें अपने एम्बेडिंग मॉडल को बड़े संग्रह के टेक्स्ट पर एक विशिष्ट तरीके से प्री-ट्रेन करना होगा। सैमान्टिक एम्बेडिंग्स को प्रशिक्षित करने के पहले तरीकों में से एक है [Word2Vec](https://en.wikipedia.org/wiki/Word2vec)। यह दो मुख्य वास्तुकलाओं पर आधारित है जिन्हें शब्दों का वितरित प्रतिनिधित्व उत्पन्न करने के लिए उपयोग किया जाता है:

 - **कंटीनुअस बैग-ऑफ-वर्ड्स** (CBoW) — इस वास्तुकला में, हम मॉडल को आसपास के संदर्भ से एक शब्द की भविष्यवाणी करने के लिए प्रशिक्षित करते हैं। दिए गए ngram $(W_{-2},W_{-1},W_0,W_1,W_2)$ के लिए, मॉडल का लक्ष्य $W_0$ को $(W_{-2},W_{-1},W_1,W_2)$ से पूर्वानुमानित करना है।
 - **कंटीनुअस स्किप-ग्राम** CBoW का उल्टा है। मॉडल वर्तमान शब्द की भविष्यवाणी करने के लिए आसपास के संदर्भ शब्दों के विंडो का उपयोग करता है।

CBoW तेज़ है, जबकि स्किप-ग्राम धीमा है, लेकिन दुर्लभ शब्दों को बेहतर प्रतिनिधित्व करता है।

![इमेज जिसमें CBoW और स्किप-ग्राम दोनों एल्गोरिदम दिखाए गए हैं जो शब्दों को वेक्टर में परिवर्तित करते हैं।](../../../../../translated_images/hi/example-algorithms-for-converting-words-to-vectors.fbe9207a726922f6.webp)

Google News डेटासेट पर प्री-ट्रेंड Word2Vec एम्बेडिंग के साथ प्रयोग करने के लिए, हम **gensim** लाइब्रेरी का उपयोग कर सकते हैं। नीचे 'neural' शब्द के सबसे समान शब्दों को पाया गया है

> **नोट:** जब आप पहली बार शब्द वेक्टर बनाते हैं, तो उन्हें डाउनलोड करने में कुछ समय लग सकता है!


In [8]:
import gensim.downloader as api
w2v = api.load('word2vec-google-news-300')

In [9]:
for w,p in w2v.most_similar('neural'):
    print(f"{w} -> {p}")

neuronal -> 0.7804799675941467
neurons -> 0.7326500415802002
neural_circuits -> 0.7252851724624634
neuron -> 0.7174385190010071
cortical -> 0.6941086649894714
brain_circuitry -> 0.6923246383666992
synaptic -> 0.6699118614196777
neural_circuitry -> 0.6638563275337219
neurochemical -> 0.6555314064025879
neuronal_activity -> 0.6531826257705688


हम शब्द से वेक्टर एम्बेडिंग भी गणना कर सकते हैं, जिसका उपयोग वर्गीकरण मॉडल को प्रशिक्षण देने में किया जाएगा (स्पष्टता के लिए हम वेक्टर के केवल पहले 20 घटकों को दिखाते हैं):


In [10]:
w2v.word_vec('play')[:20]

array([ 0.01226807,  0.06225586,  0.10693359,  0.05810547,  0.23828125,
        0.03686523,  0.05151367, -0.20703125,  0.01989746,  0.10058594,
       -0.03759766, -0.1015625 , -0.15820312, -0.08105469, -0.0390625 ,
       -0.05053711,  0.16015625,  0.2578125 ,  0.10058594, -0.25976562],
      dtype=float32)

सेमांटिकल एम्बेडिंग्स की बड़ी अच्छी बात यह है कि आप वेक्टर एन्कोडिंग को इस तरह से बदल सकते हैं कि उसका अर्थ बदल जाए। उदाहरण के लिए, हम ऐसे शब्द को खोज सकते हैं, जिसका वेक्टर प्रतिनिधित्व शब्द *king* और *woman* के जितना संभव हो नजदीक हो, और शब्द *man* से जितना हो सके दूर हो:


In [10]:
w2v.most_similar(positive=['king','woman'],negative=['man'])[0]

('queen', 0.7118192911148071)

CBoW और Skip-Grams दोनों "पूर्वानुमानित" एम्बेडिंग हैं, क्योंकि ये केवल स्थानीय संदर्भों को ध्यान में रखते हैं। Word2Vec वैश्विक संदर्भ का लाभ नहीं उठाता है। 

**FastText**, Word2Vec पर आधारित है जो प्रत्येक शब्द और प्रत्येक शब्द के अंदर पाए जाने वाले वर्ण n-grams के लिए वेक्टर प्रतिनिधित्व सीखता है। प्रतिनिधित्व के मानों को प्रत्येक प्रशिक्षण चरण में एक वेक्टर में औसतित किया जाता है। जबकि इससे प्री-ट्रेनिंग में काफी अतिरिक्त गणना जोड़ती है, यह शब्द एम्बेडिंग को उप-शब्द जानकारी को एन्कोड करने में सक्षम बनाता है। 

एक अन्य विधि, **GloVe**, सह-घटना मैट्रिक्स के विचार का लाभ उठाती है, जो सह-घटना मैट्रिक्स को अधिक अभिव्यक्तिपूर्ण और गैर-रेखीय शब्द वेक्टरों में विघटित करने के लिए न्यूरल विधियों का उपयोग करती है।

आप उदाहरण के साथ FastText और GloVe में एम्बेडिंग बदलकर खेल सकते हैं, क्योंकि gensim कई अलग-अलग शब्द एम्बेडिंग मॉडलों का समर्थन करता है।


## PyTorch में प्री-प्रशिक्षित एम्बेडिंग्स का उपयोग करना

हम ऊपर के उदाहरण को संशोधित कर सकते हैं ताकि हमारे एम्बेडिंग लेयर में मैट्रिक्स को सेमांटिकल एम्बेडिंग्स, जैसे कि Word2Vec, से पहले से भरा जा सके। हमें यह ध्यान में रखना होगा कि प्री-प्रशिक्षित एम्बेडिंग और हमारे टेक्स्ट कॉर्पस के शब्दावली संभवतः मेल नहीं खाएंगे, इसलिए हम गायब शब्दों के लिए वेट्स को यादृच्छिक मानों से इनिशियलाइज़ करेंगे:


In [11]:
embed_size = len(w2v.get_vector('hello'))
print(f'Embedding size: {embed_size}')

net = EmbedClassifier(vocab_size,embed_size,len(classes))

print('Populating matrix, this will take some time...',end='')
found, not_found = 0,0
for i,w in enumerate(vocab.get_itos()):
    try:
        net.embedding.weight[i].data = torch.tensor(w2v.get_vector(w))
        found+=1
    except:
        net.embedding.weight[i].data = torch.normal(0.0,1.0,(embed_size,))
        not_found+=1

print(f"Done, found {found} words, {not_found} words missing")
net = net.to(device)

Embedding size: 300
Populating matrix, this will take some time...Done, found 41080 words, 54732 words missing


अब चलिए अपने मॉडल को प्रशिक्षित करते हैं। ध्यान दें कि मॉडल को प्रशिक्षित करने में लगने वाला समय पिछले उदाहरण की तुलना में काफी अधिक है, क्योंकि एम्बेडिंग लेयर का आकार बड़ा है, और इसलिए पैरामीटरों की संख्या भी काफी अधिक है। इसके अलावा, इस कारण से, हमें अधिक उदाहरणों पर अपने मॉडल को प्रशिक्षित करने की आवश्यकता हो सकती है यदि हम ओवरफिटिंग से बचना चाहते हैं।


In [12]:
train_epoch_emb(net,train_loader, lr=4, epoch_size=25000)

3200: acc=0.6359375
6400: acc=0.68109375
9600: acc=0.7067708333333333
12800: acc=0.723671875
16000: acc=0.73625
19200: acc=0.7463541666666667
22400: acc=0.7560714285714286


(214.1013875559821, 0.7626759436980166)

हमारे मामले में हमें सटीकता में बड़ा बढ़ावा नहीं दिखता, जो संभवतः अलग-अलग शब्दावली के कारण है। 
अलग-अलग शब्दावली की समस्या को दूर करने के लिए, हम निम्नलिखित समाधानों में से एक का उपयोग कर सकते हैं:
* हमारे शब्दकोश पर word2vec मॉडल को फिर से प्रशिक्षित करें
* प्री-प्रशिक्षित word2vec मॉडल से शब्दावली के साथ हमारे डेटा सेट को लोड करें। डेटा सेट लोड करते समय उपयोग की जाने वाली शब्दावली निर्दिष्ट की जा सकती है।

बाद वाला तरीका आसान लगता है, खासकर क्योंकि PyTorch `torchtext` फ्रेमवर्क में एम्बेडिंग के लिए अंतर्निर्मित समर्थन होता है। उदाहरण के लिए, हम निम्न तरीके से GloVe-आधारित शब्दावली को स्थापित कर सकते हैं:


In [14]:
vocab = torchtext.vocab.GloVe(name='6B', dim=50)

100%|█████████▉| 399999/400000 [00:15<00:00, 25411.14it/s]


Loaded vocabulary has the following basic operations:
* `vocab.stoi` dictionary allows us to convert word into its dictionary index
* `vocab.itos` does the opposite - converts number into word
* `vocab.vectors` is the array of embedding vectors, so to get the embedding of a word `s` we need to use `vocab.vectors[vocab.stoi[s]]`

Here is the example of manipulating embeddings to demonstrate the equation **kind-man+woman = queen** (I had to tweak the coefficient a bit to make it work):


In [15]:
# get the vector corresponding to kind-man+woman
qvec = vocab.vectors[vocab.stoi['king']]-vocab.vectors[vocab.stoi['man']]+1.3*vocab.vectors[vocab.stoi['woman']]
# find the index of the closest embedding vector 
d = torch.sum((vocab.vectors-qvec)**2,dim=1)
min_idx = torch.argmin(d)
# find the corresponding word
vocab.itos[min_idx]

'queen'

उन एम्बेडिंग्स का उपयोग करके क्लासिफायर को ट्रेन करने के लिए, हमें पहले अपने डेटासेट को GloVe शब्दावली का उपयोग करके एन्कोड करना होगा:


In [16]:
def offsetify(b):
    # first, compute data tensor from all sequences
    x = [torch.tensor(encode(t[1],voc=vocab)) for t in b] # pass the instance of vocab to encode function!
    # now, compute the offsets by accumulating the tensor of sequence lengths
    o = [0] + [len(t) for t in x]
    o = torch.tensor(o[:-1]).cumsum(dim=0)
    return ( 
        torch.LongTensor([t[0]-1 for t in b]), # labels
        torch.cat(x), # text 
        o
    )

जैसा कि हमने ऊपर देखा है, सभी वेक्टर एम्बेडिंग्स `vocab.vectors` मैट्रिक्स में संग्रहीत हैं। यह एम्बेडिंग लेयर के वज़न में उन वज़नों को सरल कॉपी करने से लोड करना बहुत आसान बनाता है:


In [17]:
net = EmbedClassifier(len(vocab),len(vocab.vectors[0]),len(classes))
net.embedding.weight.data = vocab.vectors
net = net.to(device)

अब आइए अपना मॉडल प्रशिक्षित करते हैं और देखते हैं कि क्या हमें बेहतर परिणाम मिलते हैं:


In [18]:
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=16, collate_fn=offsetify, shuffle=True)
train_epoch_emb(net,train_loader, lr=4, epoch_size=25000)

3200: acc=0.6271875
6400: acc=0.68078125
9600: acc=0.7030208333333333
12800: acc=0.71984375
16000: acc=0.7346875
19200: acc=0.7455729166666667
22400: acc=0.7529464285714286


(35.53972978646833, 0.7575175943698017)

उन कारणों में से एक है कि हम सटीकता में महत्वपूर्ण वृद्धि नहीं देख रहे हैं, क्योंकि हमारे डेटासेट के कुछ शब्द प्री-ट्रेंड GloVe शब्दकोश में मौजूद नहीं हैं, और इसलिए उन्हें मूलतः नजरअंदाज किया जाता है। इस तथ्य को पार पाने के लिए, हम अपने डेटासेट पर अपने स्वयं के एम्बेडिंग्स प्रशिक्षित कर सकते हैं। 


## संदर्भात्मक एम्बेडिंग्स

पारंपरिक प्रीट्रेंड एम्बेडिंग प्रतिनिधित्व जैसे कि Word2Vec की एक प्रमुख सीमा शब्दार्थ अस्पष्टता की समस्या है। जबकि प्रीट्रेंड एम्बेडिंग्स संदर्भ में शब्दों के कुछ अर्थ को पकड़ सकते हैं, एक शब्द के हर संभव अर्थ को उसी एम्बेडिंग में एन्कोड किया जाता है। इससे डाउनस्ट्रीम मॉडलों में समस्याएं हो सकती हैं, क्योंकि कई शब्द जैसे 'play' शब्द का अर्थ अलग-अलग संदर्भों पर निर्भर करता है जिनमें वे उपयोग किए जाते हैं।

उदाहरण के लिए, 'play' शब्द दो अलग-अलग वाक्यों में काफी अलग अर्थ रखता है:
- मैं थिएटर में एक **play** देखने गया था।
- जॉन अपने दोस्तों के साथ **play** करना चाहता है।

उपरोक्त प्रीट्रेंड एम्बेडिंग्स 'play' शब्द के इन दोनों अर्थों को एक ही एम्बेडिंग में दर्शाते हैं। इस सीमा को पार करने के लिए, हमें **भाषा मॉडल** के आधार पर एम्बेडिंग्स बनानी होंगी, जो एक बड़े कॉर्पस पर प्रशिक्षित होता है, और *जानता है* कि शब्दों को विभिन्न संदर्भों में कैसे रखा जा सकता है। संदर्भात्मक एम्बेडिंग पर चर्चा इस ट्यूटोरियल के दायरे से बाहर है, लेकिन हम अगले इकाई में भाषा मॉडलों की बात करते समय इसका पुनः उल्लेख करेंगे।


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**अस्वीकरण**:
इस दस्तावेज़ का अनुवाद AI अनुवाद सेवा [Co-op Translator](https://github.com/Azure/co-op-translator) का उपयोग करके किया गया है। जबकि हम सटीकता के लिए प्रयास करते हैं, कृपया ध्यान दें कि स्वचालित अनुवादों में त्रुटियाँ या अशुद्धियाँ हो सकती हैं। मूल दस्तावेज़ अपनी मूल भाषा में ही प्रामाणिक स्रोत माना जाना चाहिए। महत्वपूर्ण जानकारी के लिए, पेशेवर मानव अनुवाद की सिफारिश की जाती है। इस अनुवाद के उपयोग से उत्पन्न किसी भी गलतफहमी या गलत व्याख्या के लिए हम उत्तरदायी नहीं हैं।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
